# 14 · How we build a Field-Interview Helper (B3)

**Reproduces**: [lius.cc/llm/demos/field-interview-helper](https://lius.cc/llm/demos/field-interview-helper)

**Idea**: paste a field-interview transcript (≤ 5,000 chars) → LLM automatically tags 8 categories of Daoist / folk-religion entities (deity, ritual, scripture, sect, concept, location, person, taboo) and suggests lius.cc node URLs for the top 5.

**Why it matters**: a single field researcher used to spend hours manually marking up transcripts; this tool turns a 1-hour audio interview into a publication-ready annotated text in under 30 seconds.

**Privacy**: text-only — do NOT upload audio files; pre-anonymise transcripts before submitting.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lius-cc/lius-cookbook/blob/main/notebooks/how-we-build-field-interview-helper.ipynb)

📜 CC0 1.0 · DOI [10.5281/zenodo.20248697](https://doi.org/10.5281/zenodo.20248697)

## 1 · Endpoint contract

```http
POST https://lius.cc/api/llm/generate
Content-Type: application/json

{
  "task": "tag",
  "transcript": "... up to 5,000 chars of Chinese transcript ..."
}
```

Returns:
```json
{
  "ok": true,
  "task": "tag",
  "transcript_chars": 167,
  "output": "- [location] 鎮瀾宮 → 臺灣大甲著名媽祖廟宇\n- [deity] 媽祖 → 臺灣民間信仰重要海神\n- ...",
  "provider": "gemma",
  "latency_ms": 1664,
  "snapshot": "2026-05-17"
}
```

In [ ]:
import requests, re

transcript = """我是去鎮瀾宮拜媽祖，求全家平安。
我家祖傳就是拜天上聖母，每年三月初一過爐，
我們會誦《天上聖母經》三遍。這是正一道的傳統，
全真道我就不熟了。女生月事不能進主殿。"""

r = requests.post(
    "https://lius.cc/api/llm/generate",
    json={"task": "tag", "transcript": transcript},
    timeout=60,
).json()

print(f"provider={r['provider']}  latency={r['latency_ms']}ms")
print()
print(r['output'])

## 2 · Parse the output back to structured tags

Pattern: `- [type] word → note`

In [ ]:
TAG_RE = re.compile(r'^[-•·]\s*\[([a-z]+)\]\s*([^→\-]+?)\s*[→\-]\s*(.+)$', re.MULTILINE)

tags = [
    {"type": m.group(1), "word": m.group(2).strip(), "note": m.group(3).strip()}
    for m in TAG_RE.finditer(r['output'])
]

from pprint import pprint
pprint(tags)

## 3 · 8 entity types

| Code | 中文 | Examples |
| --- | --- | --- |
| deity | 神祇 | 媽祖、城隍、玄天上帝 |
| ritual | 儀式 | 三朝醮、過爐、發奏 |
| scripture | 經文 | 道德經、天上聖母經 |
| sect | 流派 | 正一道、全真道、閭山派 |
| concept | 概念 | 三魂七魄、五雷、紫府 |
| location | 聖地 | 鎮瀾宮、武當山、龍虎山 |
| person | 人物 | 張天師、葛洪、王重陽 |
| taboo | 禁忌 | 月事不入主殿、忌穿短褲 |

## Reproducibility

- **Snapshot**: 2026-05-17
- **Rate limit**: 5 req/min per IP
- **Max transcript**: 5,000 chars
- **LLM-only** path — NO RAG retrieval; LLM scans transcript directly
- Provider order: Gemma 2.5 Flash → DeepSeek V4-Flash → OpenAI proxy